In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

In [6]:
df = pd.read_csv('../data/raw/Austin_Animal_Center_Outcomes.csv')
df.head()

,Animal ID,Name,DateTime,MonthYear,Date of Birth,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A882831,*Hamilton,07/01/2023 06:12:00 PM,Jul 2023,03/25/2023,Adoption,NaN,Cat,Neutered Male,3 months,Domestic Shorthair Mix,Black/White
1,A794011,Chunk,05/08/2019 06:20:00 PM,May 2019,05/02/2017,Rto-Adopt,NaN,Cat,Neutered Male,2 years,Domestic Shorthair Mix,Brown Tabby/White
2,A776359,Gizmo,07/18/2018 04:02:00 PM,Jul 2018,07/12/2017,Adoption,NaN,Dog,Neutered Male,1 year,Chihuahua Shorthair Mix,White/Brown
3,A821648,NaN,08/16/2020 11:38:00 AM,Aug 2020,08/16/2019,Euthanasia,NaN,Other,Unknown,1 year,Raccoon,Gray
4,A720371,Moose,02/13/2016 05:59:00 PM,Feb 2016,10/08/2015,Adoption,NaN,Dog,Neutered Male,4 months,Anatol Shepherd/Labrador Retriever,Buff


In [7]:
# Remove rows where the "Name" column is "Unknown" or missing
df = df[df['Name'].str.strip() != 'Unknown']
df = df[df['Name'].notna()]  # Remove missing values

# Remove rows with missing values in the "Outcome Type" column
df = df.dropna(subset=['Outcome Type'])

# Replace "Unknown" values in "Outcome Subtype" with NaN and remove missing rows
df['Outcome Subtype'] = df['Outcome Subtype'].replace('Unknown', np.nan)
df.dropna(subset=['Outcome Subtype'], inplace=True)

# Fill missing values in "Sex upon Outcome" with the most frequent value (mode)
most_common_sex = df['Sex upon Outcome'].mode()[0]
df['Sex upon Outcome'] = df['Sex upon Outcome'].fillna(most_common_sex)

# Function to convert age information into days
def convert_age_to_days(age_str):
    if pd.isna(age_str) or not isinstance(age_str, str):
        return None  
    
    parts = age_str.split(' ')
    if len(parts) < 2:
        return None
    
    try:
        num, unit = int(parts[0]), parts[1].lower()
        if 'year' in unit:
            return num * 365
        elif 'month' in unit:
            return num * 30
        elif 'week' in unit:
            return num * 7
        elif 'day' in unit:
            return num
    except ValueError:
        return None
    
    return None

# Convert the "Age upon Outcome" column to numerical values (in days)
df['Age in Days'] = df['Age upon Outcome'].apply(convert_age_to_days)

# Remove rows with missing values after age conversion
df = df.dropna(subset=['Age in Days'])

In [8]:
# label encoding
categorical_cols = [
    'Animal Type',
    'Sex upon Outcome'
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# normalization
scaler = MinMaxScaler()
df[['Age in Days']] = scaler.fit_transform(df[['Age in Days']])

In [9]:
df.isnull().sum()

Animal ID           0
Name                0
DateTime            0
MonthYear           0
Date of Birth       0
Outcome Type        0
Outcome Subtype     0
Animal Type         0
Sex upon Outcome    0
Age upon Outcome    0
Breed               0
Color               0
Age in Days         0
dtype: int64

In [10]:
df.describe()

,Animal Type,Sex upon Outcome,Age in Days
count,42742.000000,42742.000000,42742.000000
mean,1.575687,1.875275,0.158152
std,0.534149,1.081591,0.094939
min,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.095890
50%,2.000000,2.000000,0.121212
75%,2.000000,3.000000,0.181818
max,3.000000,4.000000,1.000000


In [11]:
# save final version
df.to_csv('../data/processed/Austin_Animal_processed.csv', index=False)